In [173]:
from sqlalchemy import create_engine, inspect, text
import sqlglot


In [174]:
# Project root and LLM api client setup
from pathlib import Path
import sys

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.llm_client import query_llm


SQL DB connection

In [175]:
db_dir = Path.cwd().parent / "data" / "db"
db_files = sorted(db_dir.glob("construction*.db"))

if not db_files:
    raise FileNotFoundError("Database file matching 'construction*.db' was not found")

db_path = db_files[0]

engine = create_engine(f"sqlite:///{db_path}")

In [176]:
inspector = inspect(engine)

# Получить все таблицы
tables = inspector.get_table_names()
tables

['contractors', 'objects', 'progress', 'works']

In [177]:
# Для каждой таблицы вывести колонки
for table_name in tables:
    print(f"\n▶️ Таблица: {table_name}")
    columns = inspector.get_columns(table_name)
    for col in columns:
        print(f"  • {col['name']} | {col['type']} | nullable={col['nullable']}")

    # Внешние ключи
    fks = inspector.get_foreign_keys(table_name)
    for fk in fks:
        print(f"  ↳ FK: {fk['constrained_columns']} → {fk['referred_table']}")

    # Индексы
    indexes = inspector.get_indexes(table_name)
    for idx in indexes:
        print(f"  🔑 INDEX: {idx['name']} ({', '.join(idx['column_names'])})")


▶️ Таблица: contractors
  • id | INTEGER | nullable=True
  • name | TEXT | nullable=False
  • work_id | INTEGER | nullable=False
  ↳ FK: ['work_id'] → works

▶️ Таблица: objects
  • id | INTEGER | nullable=True
  • name | TEXT | nullable=False
  • city | TEXT | nullable=False
  • budget | REAL | nullable=False

▶️ Таблица: progress
  • id | INTEGER | nullable=True
  • work_id | INTEGER | nullable=False
  • plan_vol | REAL | nullable=False
  • fact_vol | REAL | nullable=False
  • date | TEXT | nullable=False
  ↳ FK: ['work_id'] → works

▶️ Таблица: works
  • id | INTEGER | nullable=True
  • object_id | INTEGER | nullable=False
  • work_type | TEXT | nullable=False
  • unit | TEXT | nullable=False
  ↳ FK: ['object_id'] → objects


In [178]:
with engine.connect() as conn:
    query = """
    SELECT * 
    FROM works
    JOIN contractors ON works.id = contractors.work_id
    JOIN objects ON works.object_id = objects.id
    JOIN progress ON works.id = progress.work_id
    LIMIT 5
    """

    result = conn.execute(text(query))

print(result.keys())
result.fetchall()

RMKeyView(['id', 'object_id', 'work_type', 'unit', 'id', 'name', 'work_id', 'id', 'name', 'city', 'budget', 'id', 'work_id', 'plan_vol', 'fact_vol', 'date'])


[(61, 26, 'Установка окон', 'кв.м', 61, 'ЗАО Электро-строй', 61, 26, 'Детский сад 26', 'Екатеринбург', 305165062.8153313, 1, 61, 173.1, 167.97, '2024-04-27'),
 (98, 10, 'Отопление', 'м²', 98, 'ООО Быстро-строй', 98, 10, 'Офисный центр Альфа 10', 'Уфа', 142623453.85716254, 2, 98, 414.25, 114.85, '2024-07-10'),
 (17, 20, 'Монтаж перекрытий', 'комплект', 17, 'ООО БазовыеРаботы', 17, 20, 'Гостиница 20', 'Уфа', 358724297.61244935, 3, 17, 305.25, 306.99, '2024-01-31'),
 (105, 13, 'Фундамент', 'комплект', 105, 'ООО РазноРабота', 105, 13, 'Детский сад 13', 'Москва', 240158991.92791426, 4, 105, 779.11, 183.38, '2024-04-21'),
 (141, 37, 'Установка дверей', 'тонн', 141, 'ПАО МегаСтрой', 141, 37, 'Спортивный зал 37', 'Екатеринбург', 487671502.72202486, 5, 141, 348.23, 225.7, '2024-08-04')]

In [179]:
def get_schema_from_db(inspector) -> str:
    schema_parts = {}

    for table_name in inspector.get_table_names():
        columns = [c["name"] for c in inspector.get_columns(table_name)]
        foreingn_keys = inspector.get_foreign_keys(table_name)
        for fk in foreingn_keys:
            for col in fk["constrained_columns"]:
                columns.append(
                    f"{col} (foreing keys to table '{fk['referred_table']}')"
                )
        schema_parts[table_name] = ", ".join(columns)

    return schema_parts


db_schemas = get_schema_from_db(inspector)
db_schemas

{'contractors': "id, name, work_id, work_id (foreing keys to table 'works')",
 'objects': 'id, name, city, budget',
 'progress': "id, work_id, plan_vol, fact_vol, date, work_id (foreing keys to table 'works')",
 'works': "id, object_id, work_type, unit, object_id (foreing keys to table 'objects')"}

In [180]:
def build_system_prompt(engine) -> str:
    """Построи prompt с полным списком реальных значений из БД"""

    with engine.connect() as conn:
        # Все подрядчики
        result = conn.execute(text("SELECT DISTINCT name FROM contractors"))
        contractors = [row[0] for row in result.fetchall()]

        # Все типы работ и единицы измерения
        result = conn.execute(text("SELECT DISTINCT work_type, unit FROM works"))
        work_types_unit = [
            f"{row[0]} - {row[1]}\n"
            for row in sorted(result.fetchall(), key=lambda x: x[0])
        ]

        # Все объекты
        result = conn.execute(text("SELECT DISTINCT name FROM objects"))
        objects = [row[0] for row in result.fetchall()]

        # Все города
        result = conn.execute(text("SELECT DISTINCT city FROM objects"))
        cities = [row[0] for row in result.fetchall()]

    contractors_str = ", ".join([f"{c}" for c in contractors])
    work_types_str = "".join([f"{w}" for w in work_types_unit])
    objects_str = ", ".join([f"{o}" for o in objects])
    cities_str = ", ".join([f"{c}" for c in cities])

    return contractors_str, work_types_str, objects_str, cities_str


contractors_str, work_types_str, objects_str, cities_str = build_system_prompt(engine)
contractors_str, work_types_str, objects_str, cities_str

('ООО СтройМастер, ЗАО Строящий Лучше, ЗАО Электро-строй, АО Строймонтаж, ООО Быстро-строй, ООО Новый Век, АО Профессионал, ООО ТехСтрой, АО Фундамент, ПАО МегаСтрой, ООО БазовыеРаботы, ООО РазноРабота, ЗАО Качественно, ООО Надежный Строитель, ПАО Конструкция',
 'Вентиляция - м²\nВентиляция - м³\nВентиляция - комплект\nВентиляция - км\nВентиляция - тонн\nВентиляция - кв.м\nВентиляция - пог.м\nВнешняя отделка - м³\nВнешняя отделка - кв.м\nВнешняя отделка - тонн\nВнешняя отделка - км\nВнешняя отделка - пог.м\nВнешняя отделка - шт\nВнешняя отделка - м²\nВнутренняя отделка - комплект\nВнутренняя отделка - км\nВнутренняя отделка - пог.м\nВнутренняя отделка - тонн\nВнутренняя отделка - м²\nВнутренняя отделка - шт\nВнутренняя отделка - кв.м\nВодоснабжение - м²\nВодоснабжение - кв.м\nВодоснабжение - км\nВодоснабжение - тонн\nВодоснабжение - шт\nВодоснабжение - пог.м\nВозведение стен - м²\nВозведение стен - км\nВозведение стен - пог.м\nВозведение стен - комплект\nВозведение стен - кв.м\nВозведе

LLM query

In [232]:
SYSTEM_PROMPT = f"""
Ты преобразуешь запросы на естественном языке в один корректный SQL-запрос для SQLite.

ЗАДАЧА
Сгенерируй ровно один SQL-запрос, используя только схему БД и допустимые значения ниже.
Если корректный запрос построить невозможно, верни ровно: Невозможно ответить

СХЕМА БД
{db_schemas}

ДОПУСТИМЫЕ ЗНАЧЕНИЯ
Подрядчики: {contractors_str}
Типы работ и единицы измерения: {work_types_str}
Объекты: {objects_str}
Города: {cities_str}

ФОРМАТ ОТВЕТА
Верни только SQL-запрос.
Без пояснений, markdown, кодовых блоков, комментариев и лишнего текста.
Форматируй SQL многострочно: основные секции с новой строки, поля SELECT по одному на строку.

ПРАВИЛА
1. Сначала определи гранулярность результата: по подрядчику, по объекту или по работе. Если запрос неоднозначен, выбирай одну строку на работу.
2. Выбирай базовую таблицу по гранулярности: works для работ, objects для объектов, contractors для подрядчиков.
3. Метрики из progress привязаны к work_id. Не дублируй plan_vol и fact_vol из-за one-to-many связи с contractors.
4. Если есть агрегация по plan_vol или fact_vol, обязательно используй works.unit в SELECT и GROUP BY. Не смешивай разные единицы измерения в одной сумме. progress.unit использовать нельзя.
5. Если unit нужен, а базовая таблица не works, добавь корректный JOIN с works по схеме.
6. Не добавляй дату в SELECT, GROUP BY или агрегаты, если пользователь явно не просил детализацию по датам или периодам.
7. Все строковые значения в WHERE должны браться только из допустимых значений выше.
8. Не копируй текст пользователя в WHERE напрямую, пока он не найден в допустимом списке.
9. Для неполного имени объекта используй IN (...) со всеми подходящими полными именами из списка Объекты. Оператор = разрешён только для полного точного имени объекта.
10. Разговорные названия городов сначала преобразуй в официальное название из списка Города.
11. Используй только таблицы, поля и связи из схемы, только синтаксис SQLite, без SELECT *.
12. Все неагрегированные поля из SELECT должны быть в GROUP BY.

Если любое правило нарушается, исправь запрос или верни Невозможно ответить.
"""

REVIEW_PROMPT = (
    "Проверь предыдущий SQL-запрос на соответствие SYSTEM_PROMPT. "
    "Особенно проверь: гранулярность, JOIN, отсутствие дублирования метрик, "
    "валидность значений в WHERE, IN (...) для неполного имени объекта, "
    "works.unit в SELECT и GROUP BY при агрегации объемов, отсутствие progress.unit, "
    "и отсутствие даты без явного запроса пользователя. "
    "Если запрос некорректен, исправь его. Если корректен, верни без изменения смысла. "
    "Верни ровно один SQL-запрос без объяснений, без markdown, без комментариев и без лишнего текста."
)

In [237]:
# Запрос 1: Фильтрация по нескольким параметрам (как исходный пример)
query_1 = """
Покажи все объекты в Петербурге, для подрядчика - Новый Век, по работам связанным с покраской. 
Результат должен включать город, название объекта, имя подрядчика, название работы, ед.изм, 
плановый и фактический объем работ.
"""

# Запрос 2: Совпадение план vs факт (условие сравнения)
query_2 = """
Найди все работы, где фактический объем меньше планового объема.
Результат должен включать название объекта, тип работы, единицу измерения, плановый и фактический объемы.
"""

# Запрос 3: Агрегация и группировка
query_3 = """
Каков общий плановый объем работ по каждому подрядчику?
Результат должен содержать имя подрядчика, название работы, ед.изм. и сумму всех плановых объемов его работ.
"""

# Запрос 4: Одна конкретная сущность (объект)
query_4 = """
Покажи все работы для объекта Спортивный зал в Питере.
Результат включает подрядчика, название работы, единицу измерения, текущий статус прогресса.
"""

# Запрос 5: Поиск по типу работ без привязки к конкретному подрядчику
query_5 = """
Покажи все работы по типу связанными с кровлей, по всем объектам в Екатеринбурге.
Результат должен содержать название объекта, подрядчика, единицу измерения и объемы работ.
"""

In [238]:
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": query_3},
]

In [239]:
import re


def strip_markdown_sql(text: str) -> str:
    """Возвращает чистый SQL, даже если модель обернула его в markdown."""
    cleaned_text = text.strip()
    fenced_match = re.search(
        r"```(?:sql)?\s*\n?(.*?)```",
        cleaned_text,
        re.DOTALL | re.IGNORECASE,
    )
    if fenced_match:
        return fenced_match.group(1).strip()
    return cleaned_text


try:
    sql_query = query_llm(messages)
    sql_clean = strip_markdown_sql(sql_query)
    formatted_draft_sql = sqlglot.transpile(
        sql_clean,
        read="sqlite",
        write="sqlite",
        pretty=True,
    )[0]

    messages.append({"role": "assistant", "content": formatted_draft_sql})
    messages.append({"role": "user", "content": REVIEW_PROMPT})

    llm_response = query_llm(messages)
    llm_response_clean = strip_markdown_sql(llm_response)
    formatted_final_sql = sqlglot.transpile(
        llm_response_clean,
        read="sqlite",
        write="sqlite",
        pretty=True,
    )[0]
    print("Финальный SQL запрос от LLM:")
    print(formatted_final_sql)
except Exception as err:
    print(err)
    error_fix_query = (
        f"Предыдущий SQL-запрос вызвал ошибку выполнения: {err}. "
        "Исправь SQL-запрос так, чтобы он соответствовал SYSTEM_PROMPT. "
        "Проверь валидность фильтрации, works.unit при агрегации объемов, отсутствие progress.unit "
        "и отсутствие даты без явного запроса пользователя. "
        "Верни ровно один исправленный SQL-запрос без объяснений, без markdown, без комментариев и без лишнего текста."
    )
    messages.append({"role": "assistant", "content": error_fix_query})

    llm_response = query_llm(messages)
    llm_response_clean = strip_markdown_sql(llm_response)
    formatted_final_sql = sqlglot.transpile(
        llm_response_clean,
        read="sqlite",
        write="sqlite",
        pretty=True,
    )[0]
    print("Исправленный SQL запрос от LLM:")
    print(formatted_final_sql)


Финальный SQL запрос от LLM:
SELECT
  c.name,
  w.work_type,
  w.unit,
  SUM(p.plan_vol)
FROM contractors AS c
INNER JOIN works AS w
  ON c.work_id = w.id
INNER JOIN progress AS p
  ON w.id = p.work_id
GROUP BY
  c.name,
  w.work_type,
  w.unit


In [229]:
def execute_sql_query(engine, messages: list[dict]):
    """Генерирует, проверяет и выполняет SQL запрос."""
    try:
        working_messages = list(messages)

        draft_sql = query_llm(working_messages)
        draft_sql_clean = strip_markdown_sql(draft_sql)
        formatted_draft_sql = sqlglot.transpile(
            draft_sql_clean,
            read="sqlite",
            write="sqlite",
            pretty=True,
        )[0]
        working_messages.append({"role": "assistant", "content": formatted_draft_sql})
        working_messages.append({"role": "user", "content": REVIEW_PROMPT})

        final_sql = query_llm(working_messages)
        final_sql_clean = strip_markdown_sql(final_sql)
        allowed_query = sqlglot.transpile(
            final_sql_clean,
            read="sqlite",
            write="sqlite",
            pretty=True,
        )[0]

        with engine.connect() as conn:
            result = conn.execute(text(allowed_query))
        return result.fetchall()
    except Exception as err:
        return f"Ошибка при выполнении SQL запроса: {err}"


In [231]:
execute_sql_query(engine, messages)

[('АО Профессионал', 'кв.м', 4810.38),
 ('АО Профессионал', 'км', 3557.42),
 ('АО Профессионал', 'комплект', 257.99),
 ('АО Профессионал', 'м²', 638.58),
 ('АО Профессионал', 'м³', 532.18),
 ('АО Профессионал', 'пог.м', 1195.83),
 ('АО Профессионал', 'тонн', 3017.9),
 ('АО Строймонтаж', 'кв.м', 1929.5),
 ('АО Строймонтаж', 'км', 547.69),
 ('АО Строймонтаж', 'комплект', 1106.81),
 ('АО Строймонтаж', 'м²', 6094.55),
 ('АО Строймонтаж', 'м³', 2444.37),
 ('АО Строймонтаж', 'пог.м', 2603.35),
 ('АО Строймонтаж', 'тонн', 1973.59),
 ('АО Фундамент', 'кв.м', 837.14),
 ('АО Фундамент', 'км', 1715.31),
 ('АО Фундамент', 'комплект', 2253.46),
 ('АО Фундамент', 'пог.м', 3684.91),
 ('АО Фундамент', 'тонн', 3739.98),
 ('АО Фундамент', 'шт', 1481.82),
 ('ЗАО Качественно', 'кв.м', 4954.45),
 ('ЗАО Качественно', 'км', 10518.24),
 ('ЗАО Качественно', 'комплект', 856.05),
 ('ЗАО Качественно', 'м²', 1489.23),
 ('ЗАО Качественно', 'м³', 2084.36),
 ('ЗАО Качественно', 'пог.м', 1473.1200000000001),
 ('ЗАО Ка